## Overview 
Welcome to EngageMetrics' Digital Marketing team! We're investigating how different marketing channels impact user engagement on our platform. In this lab, you'll analyze real user interaction data to help optimize our marketing strategy. Using Python's statistical testing libraries, you'll compare engagement metrics across different marketing channels to determine which drives the highest user engagement.

This hands-on experience mirrors the actual analysis process our team uses to make data-driven marketing decisions. The skills you'll practice are essential for any data scientist working on conversion optimization and user engagement analysis.

## Learning Outcomes 
- Conduct exploratory analysis on A/B test data using Python
- Apply statistical testing to compare user engagement metrics
- Create visualizations to communicate A/B test results
- Draw data-driven conclusions about marketing channel effectiveness

## Dataset Information
We'll be working with the <b>AB_test_user_engagement.csv</b> dataset, which contains:
- User engagement metrics (clicks, time_spent)
- User demographics (age, location)
- Marketing channel information
- Test group assignments (A/B)
- Account details (type, plan)

## Activities 
### Activity 1: Data Exploration and Preparation 
Let's start by understanding our user engagement data.

<b>Step 1:</b> Import necessary libraries and preview data

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

<b>Step 2:</b> Adjust paths to import data

In [ ]:
BASE_DIR = Path.cwd().resolve().parents[1]
DATA_DIR = BASE_DIR / "data"
user_engagement_path = DATA_DIR / "AB_test_user_engagement.csv"

<b>Step 3:</b> Load and inspect the data

In [ ]:
df = pd.read_csv(user_engagement_path)
display(df.head())

display(df.info())

<b>Step 4:</b> Examine data quality

In [ ]:
# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# View basic statistics
print("\nBasic Statistics:")
print(df.describe())

print('\nGroup distribution')
print(df['group'].value_counts())

### Activity 2: Initial Analysis 
Compare engagement metrics between test groups to identify potential differences.

<b>Step 1:</b> Calculate baseline metrics

In [ ]:
group_metrics = df.groupby('group').agg({
    'clicks': ['mean', 'std', 'count'],
    'time_spent': ['mean', 'std', 'count']
})

print('group metrics')
print(group_metrics)

<b>Step 2:</b> Visualize distributions of clicks and time spent

In [ ]:
# Create side-by-side boxplots
# YOUR CODE HERE 
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.boxplot(x='marketing_channel', y='time_spent', hue='group', data=df)
plt.title('time spent distribution by marketing channel and group')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.boxplot(x='marketing_channel', y='clicks', hue='group', data=df)
plt.title('clicks distribution by marketing channel and group')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### Activity 3: Statistical Testing 
Now let's determine if the differences we observed are statistically significant.

<b>Step 1:</b> Formulate hypothesis

### For each marketing channel, we'll test:
#### H0: No difference in engagement between groups A and B
#### H1: There is a difference in engagement between groups A and B

<b>Step 2:</b> Conduct statistical tests

In [ ]:
# perform a t-test for each channel
test_results = []

for channel in df['marketing_channel'].unique():
    channel_data = df[df['marketing_channel'] == channel]
    group_a = channel_data[channel_data['group'] == 'A']
    group_b = channel_data[channel_data['group'] == 'B']

    clicks_a = group_a['clicks']
    clicks_b = group_b['clicks']
    time_a = group_a['time_spent']
    time_b = group_b['time_spent']

    t_stat_clicks, p_value_clicks = stats.ttest_ind(clicks_a, clicks_b)
    t_stat_time, p_value_time = stats.ttest_ind(time_a, time_b)

    test_results.append({
        'channel': channel,
        'click_p_value': p_value_clicks,
        'click_t_stat': t_stat_clicks,
        'time_p_value': p_value_time,
        'time_t_stat': t_stat_time
    })

results_df = pd.DataFrame(test_results)
print("\nStatistical Test Results:")
print(results_df)

In [ ]:
# Create summary visualization
plt.figure(figsize=(15, 6))

# Plot clicks
plt.subplot(1, 2, 1)
sns.barplot(x='marketing_channel', y='clicks', hue='group', data=df,
            ci=95, capsize=0.05)
plt.title('Average Clicks by Channel and Group')
plt.xticks(rotation=45)

# Plot time spent
plt.subplot(1, 2, 2)
sns.barplot(x='marketing_channel', y='time_spent', hue='group', data=df,
            ci=95, capsize=0.05)
plt.title('Average Time Spent by Channel and Group')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Interpreting t-test Results for Marketing Channels

In this test, two groups (A and B) were compared within each marketing channel to see if there are significant differences in two metrics: clicks and time spent. 

The t-test checks if the average values of these metrics differ enough between groups to be unlikely due to random chance.

    - The **t-statistic** tells you the size and direction of the difference between groups. A larger absolute value means a bigger difference.
    
    - The **p-value tells** you how likely it is to see such a difference if there really is no effect. A common threshold is 0.05; if the p-value is below this, the difference is considered statistically significant.

## Results:

    - For Social Media, neither clicks (p=0.106) nor time spent (p=0.264) show significant differences between groups A and B.
    
    - For Paid Ads, clicks have a significant difference (p=0.024), meaning groups A and B differ in clicks here.
      Time spent does not differ significantly (p=0.322).
    
    - For Referral, neither clicks (p=0.507) nor time spent (p=0.529) show significant differences.

In simple terms, only the Paid Ads channel shows a meaningful difference in clicks between the two groups, suggesting that whatever change was tested in group B affected clicks in Paid Ads. The other channels and the time spent metric do not show strong evidence of differences.


# interpreting these results in a business context

## Are channels independent from each other?

Usually, marketing channels like Social Media, Paid Ads, and Referral are treated as independent segments because they represent different sources or methods of reaching customers. Users exposed to one channel typically don't overlap much with users from another channel, so their behaviors can be analyzed separately.

## Are groups A and B within one channel independent from groups A and B in another channel?
    
Typically the A/B split within each channel is independent of the splits in other channels. 
Each channel’s A and B groups represent a test of a specific feature or variation unique to that channel. For example, group A and B in Social Media might test a different ad design, while group A and B in Paid Ads might test a different bidding strategy. These tests are separate experiments.

## Does A/B in one channel represent the same feature difference as A/B in another channel?

Not necessarily. The A/B test in each channel usually targets a different feature or change relevant to that channel. So, the meaning of A vs. B in Social Media is independent of the meaning of A vs. B in Paid Ads.

In business terms, this means you can interpret the test results for each channel on its own. For example, the significant difference in clicks for Paid Ads suggests that the tested change in Paid Ads had an impact, but this doesn’t imply anything about Social Media or Referral channels.